### Try an analysis

In [ ]:
import os
os.environ['PISA_RESOURCES'] = "../"
os.environ['PISA_RESOURCES'] += os.pathsep + "../../"
os.environ['PISA_RESOURCES'] += os.pathsep + "/data/ana/LE/"

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import math
import numpy as np
from uncertainties import unumpy as unp
import matplotlib.pyplot as plt
import pisa
import copy
from scipy.interpolate import make_interp_spline

from pisa.core.pipeline import Pipeline
from pisa.core.distribution_maker import DistributionMaker
from pisa.core.detectors import Detectors
from pisa import FTYPE, ureg
from pisa.utils.fileio import to_file
from pisa.analysis.analysis import update_param_values, update_param_values_detector

In [ ]:
params = {'legend.fontsize': 18,
          'figure.figsize': (9, 9*0.618),
          'axes.labelsize': 18,
          'axes.titlesize': 18,
          'xtick.labelsize': 18,
          'ytick.labelsize': 18}
plt.rcParams.update(params)

In [ ]:
%%time
#template_maker = DistributionMaker(["settings/pipeline/pipeline_upgrade_neutrinos_std_osc_NO.cfg", "settings/pipeline/pipeline_upgrade_muons.cfg"], profile=True)
#template_maker = DistributionMaker(["settings/pipeline/pipeline_oscnext_neutrinos_std_osc_NO.cfg", "settings/pipeline/pipeline_oscnext_muons.cfg"], profile=True)


p1_nu = Pipeline("settings/pipeline/pipeline_upgrade_neutrinos_std_osc_NO.cfg")
p1_mu = Pipeline("settings/pipeline/pipeline_upgrade_muons.cfg")
p2_nu = Pipeline("settings/pipeline/pipeline_oscnext_neutrinos_std_osc_NO.cfg")
#p2_mu = Pipeline("settings/pipeline/pipeline_oscnext_muons.cfg")

shared_params = list(p1_nu.params.free.names) #+ list(p1_mu.params.free.names)
shared_params.remove('icu_dom_eff')
template_maker = Detectors([p1_nu, p1_mu, p2_nu], shared_params=shared_params, profile=True)

### Fake data

In [ ]:
fake_data = template_maker.get_outputs(return_sum=True)

### Try a quick $\chi^2$ scan without changing systematics

In [ ]:
def get_mod_chi2(data, template_maker, tau_norm=None):
    params = template_maker.params
    
    if tau_norm is not None:
        params['nutau_norm'].value = tau_norm
        params['nutau_cc_norm'].value = tau_norm
        if 'nutau_norm_deepcore' in params.names:
            params['nutau_norm_deepcore'].value = tau_norm
            params['nutau_cc_norm_deepcore'].value = tau_norm
        
    if template_maker.__class__.__name__ == "Detectors":
        update_param_values_detector(template_maker, params)
        metrics, template = [], template_maker.get_outputs()
        for i in range(len(template)):
            metrics.append(data[i][0].mod_chi2(template[i]))
        return np.sum(metrics) + template_maker.params.priors_penalty('mod_chi2'), metrics
    else:
        update_param_values(template_maker, params)
        return data[0].mod_chi2(template_maker.get_outputs()) + template_maker.params.priors_penalty('mod_chi2')

In [ ]:
%%time
print(get_mod_chi2(fake_data, template_maker, 0.7))

template_maker.reset_all()
if template_maker.__class__.__name__ == "Detectors":
    update_param_values_detector(template_maker, template_maker.params)

In [ ]:
def scan_chi2(data, template_maker, gridsize=11):
    tau_norms = np.linspace(0.7, 1.3, gridsize)

    chi2_scan = np.zeros(gridsize)

    if template_maker.__class__.__name__ == "Detectors":
        chi2_scan_det1 = np.zeros(gridsize)
        chi2_scan_det2 = np.zeros(gridsize)
        for i, tn in enumerate(tau_norms):
            chi2s = get_mod_chi2(data, template_maker, tn)
            chi2_scan[i] = chi2s[0]
            chi2_scan_det1[i] = chi2s[1][0]
            chi2_scan_det2[i] = chi2s[1][1]
        return chi2_scan, chi2_scan_det1, chi2_scan_det2, tau_norms
    else:
        for i, tn in enumerate(tau_norms):
            chi2_scan[i] = get_mod_chi2(data, template_maker, tn)      
        return chi2_scan, tau_norms

In [ ]:
%%time
if template_maker.__class__.__name__ == "Detectors":
    chi2, chi2_det1, chi2_det2, tau_norms = scan_chi2(fake_data, template_maker, gridsize=11)
else:
    chi2, tau_norms = scan_chi2(fake_data, template_maker, gridsize=11)

template_maker.reset_all()
if template_maker.__class__.__name__ == "Detectors":
    update_param_values_detector(template_maker, template_maker.params)

In [ ]:
#save scan
if template_maker.__class__.__name__ == "Detectors":
    d = {'chi2': chi2, 'chi2_det1': chi2_det1, 'chi2_det2': chi2_det2, 'tau_norms': tau_norms}
else:
    d = {'chi2': chi2, 'tau_norms': tau_norms}

with open("../scans/quick_scan_tau_dynedge.pkl", "wb") as f:
    pickle.dump(d, f)

#### plot contours

In [ ]:
with open("../scans/quick_scan_tau_dynedge.pkl", "rb") as f:
    d = pickle.load(f)

tau_norms = d['tau_norms']

if template_maker.__class__.__name__ == "Detectors":
    chi2 = d['chi2']
    chi2_det1 = d['chi2_det1']
    chi2_det2 = d['chi2_det2']
else:
    chi2 = d['chi2']
    
with open("../scans/quick_scan_tau_DC.pkl", "rb") as f:
    d = pickle.load(f)
chi2_DC = d['chi2']

In [ ]:
interp = make_interp_spline(tau_norms, chi2, k=3)

interp_xs = np.linspace(tau_norms[0], tau_norms[-1], 100)
interp_ys = interp(interp_xs)

if template_maker.__class__.__name__ == "Detectors":
    interp_det1 = make_interp_spline(tau_norms, chi2_det1, k=3)
    interp_det2 = make_interp_spline(tau_norms, chi2_det2, k=3)
    
    interp_ys_det1 = interp_det1(interp_xs)
    interp_ys_det2 = interp_det2(interp_xs)
    
interp_DC = make_interp_spline(tau_norms, chi2_DC, k=3)
interp_ys_DC = interp_DC(interp_xs)

In [ ]:
fig = plt.figure(figsize=(9, 9*0.618))

if template_maker.__class__.__name__ == "Detectors": 
    plt.plot(interp_xs, interp_ys_det2, label='OscNext (12yr)', linestyle='--', color='Tab:orange')
    plt.plot(interp_xs, interp_ys_det1, label='Upgrade(d) (3yr)', linestyle='--', color='Tab:blue')
    
plt.plot(interp_xs, interp_ys, label='Combined', color='black', linewidth=2)

plt.plot(interp_xs, interp_ys_DC, label='OscNext (15yr)', color='black', linewidth=2, linestyle=':')

plt.axvline(template_maker.params.nutau_norm.nominal_value.magnitude, color='r', label='Injected \ntruth')

plt.xlabel(r'$\nu_{\tau}$ norm')
plt.ylabel('$\chi^{2}_{mod}$', fontsize=16)
plt.title('Fixed systematics')
plt.legend(prop={'size':14}, loc='upper right') #, bbox_to_anchor = [1.1, 1.1]

#plt.savefig('../plots/quick_scan_tau_dynedge.png', bbox_inches='tight')

### livetime

In [ ]:
template_maker.params.livetime = 3.0 * ureg.common_years
fake_data = template_maker.get_outputs(return_sum=True)

if template_maker.__class__.__name__ == "Detectors":
    chi2, chi2_det1, chi2_det2, tau_norms = scan_chi2(fake_data, template_maker, gridsize=11)
else:
    chi2, tau_norms = scan_chi2(fake_data, template_maker, gridsize=11)

template_maker.reset_all()
if template_maker.__class__.__name__ == "Detectors":
    update_param_values_detector(template_maker, template_maker.params)
    
chi2_15y = np.array(chi2)

In [ ]:
for i in range(1, 16):
    eval('plt.plot(tau_norms, chi2_%iy)'%(i))